In [ ]:
!mamba install -y transformers datasets evaluate
!mamba install -c scikit-learn
!mamba install -y  "accelerate"

In [5]:
import os, random
import numpy as np

import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer)

In [26]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

In [27]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████████████████████████████████████████████████████████| 50000/50000 [00:22<00:00, 2243.22 examples/s]


In [28]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [29]:
training_args = TrainingArguments(
    output_dir="./results",
    use_cpu = True, # GPU를 사용하지 않음을 명시
    per_device_train_batch_size=4, # 메모리 부족 방지를 위해 작게 설정
    gradient_accumulation_steps=4, # 배치 사이즈가 작으니 gradient accumulation 활용
    eval_strategy="epoch",
    eval_steps=500,
    num_train_epochs=1
)

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"]
)
